# Dent-FINE Dental Neck 学习笔记：DUG-HOIF / D-SWFM / D-CED

这个 notebook 对应当前仓库中接入的 dental neck 模块：

- `DUG-HOIF`: Dental Uncertainty-Guided High-Order Interleaved Fusion
- `D-SWFM`: Dental Spectral-Wavelet Feature Modulation
- `D-CED`: Dental-aware Cascaded Encoder-Decoder Fusion

目标不是直接训练，而是像 `dentfine_useful_modules_extracted_shape_notes.ipynb` 一样，用**代码片段、shape 检查、参数量检查和消融配置表**帮助你理解这些模块为什么这样接入，以及每个消融实验在测什么。

## 0. 学习路线

建议按这个顺序看：

1. 先看 `baseline -> dug_* -> dswfm -> dced_* -> full_dental_neck` 的消融链路。
2. 再看 DUG-HOIF 的 5 个子开关：`interleave / multibranch / routing / high_order / omni_gate`。
3. 然后看 D-SWFM 如何在频域和多尺度细节之间做调制。
4. 最后看 D-CED 如何做 top-down / bottom-up 级联融合，以及为什么 `dced_lite` 和 `dced_dug_hoif` 要分开测。

注意：这里的代码探针默认用随机张量，目的是检查 shape、参数量和信息流，不代表真实 AP。

In [2]:
from pathlib import Path
import sys
import inspect
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

REPO_ROOT = Path('/home/wuxi/Dent-FINE')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine.deim.hybrid_encoder import (
    DentalUncertaintyGuidedHOIF,
    DentalSpectralWaveletRefinerV2,
    DentalCEDCascadeEncoderDecoderV2,
    HybridEncoder,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo:', REPO_ROOT)
print('device:', DEVICE)

/home/wuxi/miniconda3/envs/deimv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo: /home/wuxi/Dent-FINE
device: cuda


/home/wuxi/miniconda3/envs/deimv2/lib/python3.11/site-packages/pytorch_wavelets/dtcwt/coeffs.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


## 1. 串行消融链路总览

这 10 个实验不是随便排列的，而是按“从基础注入到完整 neck”的增量路径设计：

| 实验 | 目的 | 关键开关 |
|---|---|---|
| `baseline` | 原始 DEIM/D-FINE neck | 全部关闭 |
| `dug_interleave` | 只测 pair-wise 双向注入 | DUG on, multibranch off |
| `dug_multibranch` | 加 local/context/edge 三分支 | routing/high_order/omni off |
| `dug_routing` | 加不确定性路由 | high_order/omni off |
| `dug_high_order` | 加二/三阶交互 | omni off |
| `dug_full` | 完整 DUG-HOIF | DUG 全开 |
| `dswfm` | 单独测频域/小波细节调制 | D-SWFM on |
| `dced_lite` | 单独测轻量 CED 级联 | D-CED on, DUG-in-CED off |
| `dced_dug_hoif` | 测 CED 内使用 DUG fusion | D-CED on, stable DUG-in-CED |
| `full_dental_neck` | 稳定组合版 dental neck | stable DUG + D-SWFM + D-CED-lite |

In [3]:
DUG_ABLATIONS = [
    ('baseline',        dict(use_dug_hoif_fusion=False)),
    ('dug_interleave',  dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=False)),
    ('dug_multibranch', dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                             dug_hoif_use_uncertainty_routing=False, dug_hoif_use_high_order=False, dug_hoif_use_omni_gate=False)),
    ('dug_routing',     dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                             dug_hoif_use_uncertainty_routing=True, dug_hoif_use_high_order=False, dug_hoif_use_omni_gate=False)),
    ('dug_high_order',  dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                             dug_hoif_use_uncertainty_routing=True, dug_hoif_use_high_order=True, dug_hoif_use_omni_gate=False)),
    ('dug_full',        dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                             dug_hoif_use_uncertainty_routing=True, dug_hoif_use_high_order=True, dug_hoif_use_omni_gate=True)),
]

for name, cfg in DUG_ABLATIONS:
    print(f'{name:16s}', cfg)

baseline         {'use_dug_hoif_fusion': False}
dug_interleave   {'use_dug_hoif_fusion': True, 'dug_hoif_use_interleave': True, 'dug_hoif_use_multibranch': False}
dug_multibranch  {'use_dug_hoif_fusion': True, 'dug_hoif_use_interleave': True, 'dug_hoif_use_multibranch': True, 'dug_hoif_use_uncertainty_routing': False, 'dug_hoif_use_high_order': False, 'dug_hoif_use_omni_gate': False}
dug_routing      {'use_dug_hoif_fusion': True, 'dug_hoif_use_interleave': True, 'dug_hoif_use_multibranch': True, 'dug_hoif_use_uncertainty_routing': True, 'dug_hoif_use_high_order': False, 'dug_hoif_use_omni_gate': False}
dug_high_order   {'use_dug_hoif_fusion': True, 'dug_hoif_use_interleave': True, 'dug_hoif_use_multibranch': True, 'dug_hoif_use_uncertainty_routing': True, 'dug_hoif_use_high_order': True, 'dug_hoif_use_omni_gate': False}
dug_full         {'use_dug_hoif_fusion': True, 'dug_hoif_use_interleave': True, 'dug_hoif_use_multibranch': True, 'dug_hoif_use_uncertainty_routing': True, 'dug_hoif_us

## 2. DUG-HOIF 的核心思想

DUG-HOIF 用在 FPN/PAN 的 pair-wise fusion 位置，替代简单 concat：

```text
upsample/high feature + low feature -> DUG-HOIF -> RepNCSPELAN4 / CSPLayer
```

它由 5 个可消融子组件组成：

- `interleave`: 高低层特征互相注入，避免单纯 concat 后再让卷积自己学习对齐。
- `multibranch`: local / context / edge 三分支，分别偏向局部纹理、宽感受野语义、边界细节。
- `uncertainty_routing`: 用局部方差作为不确定性提示，动态分配分支权重。
- `high_order`: 引入二阶/三阶乘性交互，增强牙齿结构和边界的组合表达。
- `omni_gate`: channel / spatial / coordinate 三类门控，做全轴校准。

经验上，`interleave + multibranch` 通常最稳；`routing/high_order/omni` 需要更谨慎的初始化和更强正则，否则可能变成过强扰动。

In [4]:
print(inspect.getsource(DentalUncertaintyGuidedHOIF)[:9000])

class DentalUncertaintyGuidedHOIF(nn.Module):
    """
    DUG-HOIF: Dental Uncertainty-Guided High-Order Interleaved Fusion.

    It supports both single-level enhancement and pair-wise FPN/PAN fusion.
    Ablation flags keep the same module class usable from interleave-only to full
    uncertainty-routed high-order omni-axis recalibration.
    """
    def __init__(
        self,
        channels,
        act='silu',
        reduction=8,
        route_branches=4,
        res_init=-2.0,
        use_interleave=True,
        use_multibranch=True,
        use_uncertainty_routing=True,
        use_high_order=True,
        use_omni_gate=True,
    ):
        super().__init__()
        assert route_branches == 4, "DUG-HOIF uses local/context/edge/high_order branches."

        hidden = max(channels // reduction, 16)
        self.channels = channels
        self.use_interleave = bool(use_interleave)
        self.use_multibranch = bool(use_multibranch)
        self.use_uncertainty_routing = bool

In [5]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

x = torch.randn(2, 64, 40, 40).to(DEVICE)
y = torch.randn(2, 64, 40, 40).to(DEVICE)

DUG_UNIT_CONFIGS = {
    'interleave': dict(use_interleave=True, use_multibranch=False),
    'multibranch': dict(use_interleave=True, use_multibranch=True, use_uncertainty_routing=False, use_high_order=False, use_omni_gate=False),
    'routing': dict(use_interleave=True, use_multibranch=True, use_uncertainty_routing=True, use_high_order=False, use_omni_gate=False),
    'high_order': dict(use_interleave=True, use_multibranch=True, use_uncertainty_routing=True, use_high_order=True, use_omni_gate=False),
    'full': dict(use_interleave=True, use_multibranch=True, use_uncertainty_routing=True, use_high_order=True, use_omni_gate=True),
}

with torch.no_grad():
    for name, kwargs in DUG_UNIT_CONFIGS.items():
        m = DentalUncertaintyGuidedHOIF(64, **kwargs).to(DEVICE).eval()
        out = m(x, y)
        delta = (out - x).abs().mean().item()
        print(f'{name:12s} params={count_params(m)/1e6:.3f}M shape={tuple(out.shape)} mean|out-x|={delta:.4f}')

RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 3. D-SWFM: 频域 + 细节跨层传递

D-SWFM 不是 pair-wise fusion，而是对一组 FPN/PAN 特征做整体 refine：

```text
[P3, P4, P5] -> spectral modulation -> fine-to-coarse detail transfer -> optional coarse-to-fine semantic compensation
```

它包含三个可消融方向：

- `dswfm_use_dynamic_fourier`: 动态频带权重，随输入图像/特征变化。
- `dswfm_use_haar_detail`: 用 Haar-like 细节幅值增强高频细节提示。
- `dswfm_use_bidirectional`: 除了细节从浅层到深层，还把深层语义补偿回浅层。

当前仓库中更推荐把 D-SWFM 放在 `paper_refine_apply=proj`，也就是 projected features 上；如果放在 `outs`，会直接扰动 decoder 输入，可能更不稳。

In [6]:
print(inspect.getsource(DentalSpectralWaveletRefinerV2)[:9000])

class DentalSpectralWaveletRefinerV2(nn.Module):
    """
    D-SWFM: Dental Spectral-Wavelet Feature Modulation.

    It first applies adaptive Fourier band modulation per feature level, then
    transfers fine detail to coarser levels and optionally sends coarse semantic
    compensation back to finer levels.
    """
    def __init__(
        self,
        channels,
        num_levels=3,
        res_init=-2.0,
        act='silu',
        use_bidirectional=True,
        use_dynamic_fourier=True,
        use_haar_detail=True,
    ):
        super().__init__()
        self.num_levels = int(num_levels)
        self.use_bidirectional = bool(use_bidirectional)
        self.use_haar_detail = bool(use_haar_detail)

        self.spectral = nn.ModuleList([
            AdaptiveFourierSpectralModulationV2(
                channels,
                res_init=res_init,
                act=act,
                dynamic=use_dynamic_fourier,
            )
            for _ in range(self.num_levels)
    

In [ ]:
feats = [
    torch.randn(2, 64, 80, 80).to(DEVICE),
    torch.randn(2, 64, 40, 40).to(DEVICE),
    torch.randn(2, 64, 20, 20).to(DEVICE),
]

with torch.no_grad():
    for name, kwargs in {
        'full': dict(use_dynamic_fourier=True, use_haar_detail=True, use_bidirectional=True),
        'static_fourier': dict(use_dynamic_fourier=False, use_haar_detail=True, use_bidirectional=True),
        'highpass_only': dict(use_dynamic_fourier=True, use_haar_detail=False, use_bidirectional=True),
        'fine_to_coarse': dict(use_dynamic_fourier=True, use_haar_detail=True, use_bidirectional=False),
    }.items():
        m = DentalSpectralWaveletRefinerV2(64, num_levels=3, res_init=-2.4, **kwargs).to(DEVICE).eval()
        outs = m(feats)
        shapes = [tuple(o.shape) for o in outs]
        deltas = [(o - f).abs().mean().item() for o, f in zip(outs, feats)]
        print(f'{name:14s} params={count_params(m)/1e6:.3f}M shapes={shapes} delta={deltas}')

## 4. D-CED: 级联 Encoder-Decoder 融合

D-CED 的作用是把多尺度特征再做一次 top-down / bottom-up 的级联融合：

```text
[P3, P4, P5]
  -> top-down: P5 -> P4 -> P3
  -> bottom-up: P3 -> P4 -> P5
  -> residual output
```

两个关键实验：

- `dced_lite`: 只用 normalized base fusion，不在 CED 内部嵌 DUG，参数和扰动更可控。
- `dced_dug_hoif`: 在 CED 的 pair-wise fusion unit 中使用 DUG-HOIF，表达力更强，但更容易过重。

组合 `full_dental_neck` 时建议避免“双重 full DUG”：FPN/PAN 已经用了 DUG，就不要在 D-CED 里再放 full DUG，否则参数和门控扰动会一起膨胀。

In [7]:
print(inspect.getsource(DentalCEDCascadeEncoderDecoderV2))

class DentalCEDCascadeEncoderDecoderV2(nn.Module):
    """D-CED: Dental-aware Cascaded Encoder-Decoder Fusion."""
    def __init__(
        self,
        channels,
        num_levels=3,
        stages=2,
        expansion=1.0,
        depth_mult=1.0,
        res_init=-1.8,
        version='dfine',
        act='silu',
        use_dug_hoif=True,
        dug_kwargs=None,
    ):
        super().__init__()
        self.stages = nn.ModuleList([
            DentalCEDCascadeStageV2(
                channels,
                num_levels=num_levels,
                expansion=expansion,
                depth_mult=depth_mult,
                res_init=res_init,
                version=version,
                act=act,
                use_dug_hoif=use_dug_hoif,
                dug_kwargs=dug_kwargs,
            )
            for _ in range(max(int(stages), 1))
        ])

    def forward(self, feats, verbose=False):
        for stage in self.stages:
            feats = stage(feats, verbose=verbose)
 

In [ ]:
feats = [
    torch.randn(2, 64, 80, 80).to(DEVICE),
    torch.randn(2, 64, 40, 40).to(DEVICE),
    torch.randn(2, 64, 20, 20).to(DEVICE),
]

stable_dug_kwargs = dict(
    use_interleave=True,
    use_multibranch=True,
    use_uncertainty_routing=False,
    use_high_order=False,
    use_omni_gate=False,
    res_init=-2.0,
)

with torch.no_grad():
    for name, kwargs in {
        'dced_lite': dict(use_dug_hoif=False, dug_kwargs=stable_dug_kwargs),
        'dced_stable_dug': dict(use_dug_hoif=True, dug_kwargs=stable_dug_kwargs),
    }.items():
        m = DentalCEDCascadeEncoderDecoderV2(
            64, num_levels=3, stages=1, expansion=0.5, depth_mult=0.34,
            res_init=-2.2, use_dug_hoif=kwargs['use_dug_hoif'], dug_kwargs=kwargs['dug_kwargs'],
        ).to(DEVICE).eval()
        outs = m(feats)
        shapes = [tuple(o.shape) for o in outs]
        deltas = [(o - f).abs().mean().item() for o, f in zip(outs, feats)]
        print(f'{name:16s} params={count_params(m)/1e6:.3f}M shapes={shapes} delta={deltas}')

## 5. 在 HybridEncoder 里的接入位置

当前接入点可以概括为：

```text
backbone feats
  -> input_proj
  -> optional paper/refine modules on proj
  -> transformer encoder
  -> FPN top-down: concat or CMFA or DUG-HOIF
  -> PAN bottom-up: concat or CMFA or DUG-HOIF
  -> optional refine/paper modules on outs
  -> decoder
```

其中：

- `use_dug_hoif_fusion=True` 会改变 FPN/PAN pair-wise fusion 的输入通道：DUG 输出 `hidden_dim`，后续 block 不再接收 `2 * hidden_dim` concat。
- `use_dswfm=True` 和 `use_dced_cascade=True` 走 `_apply_paper_refine_stack`。
- `paper_refine_apply=proj` 更像“先校正特征再进入 neck”；`outs` 更像“直接改 decoder 输入”，风险更大。

In [ ]:
BASE_ENCODER_KWARGS = dict(
    in_channels=[256, 512, 1024],
    feat_strides=[8, 16, 32],
    hidden_dim=256,
    nhead=8,
    dim_feedforward=1024,
    num_encoder_layers=0,  # 学习/参数量检查时关闭 Transformer，避免无关计算
    expansion=0.5,
    depth_mult=0.34,
    eval_spatial_size=None,
)

EXPERIMENT_UPDATES = {
    'baseline': {},
    'dug_interleave': dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=False),
    'dug_multibranch': dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                            dug_hoif_use_uncertainty_routing=False, dug_hoif_use_high_order=False, dug_hoif_use_omni_gate=False),
    'dug_routing': dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                        dug_hoif_use_uncertainty_routing=True, dug_hoif_use_high_order=False, dug_hoif_use_omni_gate=False),
    'dug_high_order': dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                           dug_hoif_use_uncertainty_routing=True, dug_hoif_use_high_order=True, dug_hoif_use_omni_gate=False),
    'dug_full': dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                     dug_hoif_use_uncertainty_routing=True, dug_hoif_use_high_order=True, dug_hoif_use_omni_gate=True),
    'dswfm': dict(use_dswfm=True, dswfm_res_init=-2.4, paper_refine_apply='proj'),
    'dced_lite': dict(use_dced_cascade=True, dced_use_dug_hoif=False, dced_res_init=-2.2, paper_refine_apply='proj'),
    'dced_dug_hoif': dict(use_dced_cascade=True, dced_use_dug_hoif=True, dced_res_init=-2.2, paper_refine_apply='proj',
                          dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                          dug_hoif_use_uncertainty_routing=False, dug_hoif_use_high_order=False, dug_hoif_use_omni_gate=False),
    'full_dental_neck': dict(use_dug_hoif_fusion=True, dug_hoif_use_interleave=True, dug_hoif_use_multibranch=True,
                             dug_hoif_use_uncertainty_routing=False, dug_hoif_use_high_order=False, dug_hoif_use_omni_gate=False,
                             use_dswfm=True, dswfm_res_init=-2.4,
                             use_dced_cascade=True, dced_use_dug_hoif=False, dced_res_init=-2.2,
                             paper_refine_apply='proj'),
}

rows = []
for name, updates in EXPERIMENT_UPDATES.items():
    kwargs = dict(BASE_ENCODER_KWARGS)
    kwargs.update(updates)
    m = HybridEncoder(**kwargs)
    rows.append((name, count_params(m)))

for name, params in rows:
    print(f'{name:18s} encoder_params={params/1e6:.2f}M')

In [ ]:
# 一次完整 shape smoke test：模拟 B0 backbone 的 3 个输出层。
feats = [
    torch.randn(1, 256, 80, 80).to(DEVICE),
    torch.randn(1, 512, 40, 40).to(DEVICE),
    torch.randn(1, 1024, 20, 20).to(DEVICE),
]

with torch.no_grad():
    for name, updates in EXPERIMENT_UPDATES.items():
        kwargs = dict(BASE_ENCODER_KWARGS)
        kwargs.update(updates)
        m = HybridEncoder(**kwargs).to(DEVICE).eval()
        outs = m(feats)
        print(f'{name:18s}', [tuple(o.shape) for o in outs])

## 6. 查看当前仓库生成的训练配置

runner 会把每个实验写成独立 YAML，路径类似：

```text
outputs/coco_dental_neck_ablation_fixed/configs/coco/full_dental_neck.yml
```

下面的代码会优先读取 fixed 结果目录；如果还没跑，就只提示路径不存在。

In [ ]:
import yaml

CONFIG_ROOT = REPO_ROOT / 'outputs/coco_dental_neck_ablation_fixed/configs/coco'
for name in ['baseline', 'dug_multibranch', 'dced_dug_hoif', 'full_dental_neck']:
    path = CONFIG_ROOT / f'{name}.yml'
    print('\n---', name, path)
    if not path.exists():
        print('not generated yet; run tools/run_coco_dental_neck_ablation_gpu1.sh first')
        continue
    cfg = yaml.safe_load(path.read_text())
    print(json.dumps(cfg.get('HybridEncoder', {}), indent=2, ensure_ascii=False))

## 7. 读取消融结果总表

训练完成后，runner 会持续刷新：

```text
outputs/coco_dental_neck_ablation_fixed/ablation_summary.md
outputs/coco_dental_neck_ablation_fixed/ablation_summary.csv
```

如果你看到 `routing/high_order/omni` 掉点，不一定说明 DUG 全部无效；更常见的是这些增强项在当前数据规模和训练策略下扰动太强，需要更弱残差、更长 schedule 或只保留稳定子集。

In [ ]:
summary_csv = REPO_ROOT / 'outputs/coco_dental_neck_ablation_fixed/ablation_summary.csv'
if summary_csv.exists():
    try:
        import pandas as pd
        df = pd.read_csv(summary_csv)
        display(df)
    except Exception as e:
        print(summary_csv.read_text()[:4000])
        print('pandas display failed:', e)
else:
    print('summary not found:', summary_csv)

## 8. 复现实验命令

完整串行跑法：

```bash
cd /home/wuxi/Dent-FINE
tools/run_coco_dental_neck_ablation_gpu1.sh
```

只跑某几个关键实验：

```bash
EXPERIMENTS="dug_multibranch dced_lite full_dental_neck" \
  tools/run_coco_dental_neck_ablation_gpu1.sh
```

快速 dry-run 只生成配置，不启动训练：

```bash
DRY_RUN=1 EXPERIMENTS="baseline full_dental_neck" \
  tools/run_coco_dental_neck_ablation_gpu1.sh
```

默认脚本用 `CUDA_VISIBLE_DEVICES=1` 锁物理 GPU1，并把训练内部 device 写成 `cuda:0`。

## 9. 调参建议

如果后续结果仍然不明显，可以按下面优先级排查：

1. 先看 `dug_multibranch`，这是目前最稳的 DUG 子集。
2. 再看 `dced_lite`，判断 CED 级联本身是否有效。
3. 再看 `dced_dug_hoif`，但只用 stable DUG，不要直接 full DUG 嵌套。
4. `full_dental_neck` 里避免 DUG 双重叠加：FPN/PAN 用 stable DUG，D-CED 用 lite。
5. 如果 AP 下降但训练 loss 更低，可能是过拟合或 decoder 输入分布被改坏；优先减小残差、使用 `paper_refine_apply=proj`。
6. 如果 AP50 升但 AP75 降，说明定位精度受影响；优先减弱 high-order/omni 或 D-SWFM 高频强度。

一句话：牙齿任务里“边界/细节”很重要，但 neck 模块越强，越需要靠残差初始化和消融开关控制扰动。